# Silver Layer: Sales Details CDC

**Source**: `workspace.bronze.crm_sales_details_cdc`  
**Target**: `workspace.silver.crm_sales_cdc`  
**Primary Key**: Composite (`sls_ord_num`, `sls_prd_key`)

**Transformations:**
1. Trim strings
2. Clean and parse dates
3. Price corrections
4. Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length, to_date, when
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.crm_sales_details_cdc"
silver_table = "workspace.silver.crm_sales_cdc"

print("✓ Configuration loaded")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

if silver_exists:
    watermark = spark.sql(f"""
        SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01') as max_timestamp
        FROM {silver_table}
    """).collect()[0]["max_timestamp"]
    print(f"✓ Watermark: {watermark}")
else:
    watermark = "1900-01-01"
    print(f"ℹ️  Initial load")

In [0]:
df = spark.table(bronze_table).filter(col("ingestion_timestamp") > watermark)

new_count = df.count()
print(f"📊 New records: {new_count:,}")

if new_count == 0:
    dbutils.notebook.exit("No new data")

In [0]:
# Trim strings
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

print("✓ Trimmed")

In [0]:
# Clean dates - parse from yyyyMMdd integer format
df = (
    df
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

print("✓ Cleaned dates")

In [0]:
# Price corrections - calculate from sales/quantity if missing
df = df.withColumn(
    "sls_price",
    when(
        (col("sls_price").isNull()) | (col("sls_price") <= 0),
        when(
            col("sls_quantity") != 0,
            col("sls_sales") / col("sls_quantity")
        ).otherwise(None)
    ).otherwise(col("sls_price"))
)

print("✓ Corrected prices")

In [0]:
# Rename columns
rename_map = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

print("✓ Renamed columns")
df.limit(5).display()

In [0]:
if not silver_exists:
    print("Creating Silver table...")
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table))
    print(f"✓ Created: {spark.table(silver_table).count():,} records")
else:
    print("Merging into Silver...")
    target = DeltaTable.forName(spark, silver_table)
    (target.alias("target")
      .merge(
          df.alias("source"),
          "target.order_number = source.order_number AND target.product_number = source.product_number"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
    print(f"✓ MERGE complete: {spark.table(silver_table).count():,} total")